
# Datenaufbereitung — Lehrermangel und Heterogenität in deutschen Schulen

## 1. Thema
Die unmögliche Aufgabe – Warum deutsche Lehrer individuelle Förderung nicht leisten können

## 2. Fragestellung
Wie hat sich das Verhältnis zwischen wachsender Heterogenität in deutschen Schulklassen und verfügbarer Lehrerkapazität über die letzten 10–15 Jahre entwickelt – und was zeigen die Leistungsergebnisse?

## 3. Mein Beitrag: Datenaufbereitung

In diesem Notebook werden die Rohdaten aus mehreren Excel-Dateien systematisch verarbeitet:

   3.1. Laden — Einlesen der Excel-Dateien
   3.2. Bereinigen — Entfernen von Metadaten, leeren Zeilen und fehlerhaften Werten
   3.3. Transformieren — Normalisierung der Jahresangaben und Berechnung neuer Kennzahlen
   3.4. Zusammenführen — Konsolidierung aller Daten in einem sauberen Datensatz für die Visualisierung

## 4. Datenquellen

   4.1. Statistisches Bundesamt — Anzahl Lehrer und Schüler an allgemeinbildenden Schulen
   4.2. KMK (Kultusministerkonferenz) — Schüler mit sonderpädagogischer Förderung
   4.3. OECD — PISA-Studien (Mathematik, Lesekompetenz, Naturwissenschaften)

## 1. Benötigte Bibliotheken importieren

In [1]:
import pandas as pd
import numpy as np
import os

# Versionen anzeigen (zur Dokumentation)
print(f"pandas Version: {pd.__version__}")
print(f"numpy Version:  {np.__version__}")

pandas Version: 2.3.3
numpy Version:  2.3.5


## 2. Pfade zu den DatendateienAlle Excel-Dateien liegen im Ordner `Statistiken/`.Wir definieren die Pfade zentral, damit sie einfach geändert werden können.

In [2]:
# Pfad zum Ordner mit den Statistiken
DATEN_ORDNER = os.path.join("Data-Visualisation_Sem-2-main", "Statistiken")

# Dateipfade
datei_lehrer       = os.path.join(DATEN_ORDNER, "Lehrer an allgemeinbildenden Schulen.xlsx")
datei_schueler     = os.path.join(DATEN_ORDNER, "Schüler an allgemeinbildenden Schulen.xlsx")
datei_inklusion    = os.path.join(DATEN_ORDNER, "Schüler mit sonderpädagogischer Förderung in allgemeinen Schulen bis 2023.xlsx")
datei_pisa_mathe   = os.path.join(DATEN_ORDNER, "Pisa - Mathematik.xlsx")
datei_pisa_lesen   = os.path.join(DATEN_ORDNER, "Pisa - Lesekompetenz.xlsx")
datei_pisa_nawi    = os.path.join(DATEN_ORDNER, "Pisa - Naturwissenschaften.xlsx")

print("Alle Pfade definiert ✓")

Alle Pfade definiert ✓


## 6. Hilfsfunktion zum Laden der Statista-Dateien

Alle Excel-Dateien von Statista haben die gleiche Struktur:

   6.1. Sheet `'Übersicht'` enthält Metadaten (uninteressant für uns)
   6.2. Sheet `'Daten'` enthält die echten Daten, aber mit 5 leeren Zeilen am Anfang und einer leeren ersten Spalte

Statt für jede Datei den gleichen Code zu schreiben, definieren wir eine **Funktion**, 
die wir wiederverwenden können (DRY-Prinzip: *Don't Repeat Yourself*).

In [13]:
def lade_statista_datei(pfad, jahr_spalte, wert_spalte):
    """
    Lädt eine Statista-Excel-Datei und gibt einen sauberen DataFrame zurück.
    
    Parameter:
        pfad (str):         Pfad zur Excel-Datei
        jahr_spalte (str):  Name der Jahr-Spalte im Ergebnis
        wert_spalte (str):  Name der Wert-Spalte im Ergebnis
    
    Rückgabe:
        DataFrame mit zwei Spalten: [jahr_spalte, wert_spalte]
    """
    # Sheet 'Daten' laden, die ersten 5 Zeilen überspringen (Titel + Leerzeilen)
    df = pd.read_excel(pfad, sheet_name='Daten', header=None, skiprows=5)
    
    # Nur Spalten 1 und 2 behalten (Spalte 0 ist immer leer)
    df = df.iloc[:, [1, 2]].copy()
    
    # Spalten umbenennen
    df.columns = [jahr_spalte, wert_spalte]
    
    # Leere Zeilen entfernen
    df = df.dropna()
    
    return df

## 4. Daten laden

### 4.1 Lehreranzahl

In [4]:
lehrer_df = lade_statista_datei(datei_lehrer, "Schuljahr", "Anzahl_Lehrer")
print(f"Anzahl Datenzeilen: {len(lehrer_df)}")
lehrer_df.head()

Anzahl Datenzeilen: 15


,Schuljahr,Anzahl_Lehrer
0,2010/11,763988
1,2011/12,760491
2,2012/13,759136
3,2013/14,752830
4,2014/15,752358


### 4.2 Schüleranzahl

In [5]:
schueler_df = lade_statista_datei(datei_schueler, "Schuljahr", "Anzahl_Schueler")
print(f"Anzahl Datenzeilen: {len(schueler_df)}")
schueler_df.head()

Anzahl Datenzeilen: 33


,Schuljahr,Anzahl_Schueler
0,1992/93,9344364
1,1993/94,9557729
2,1994/95,9759711
3,1995/96,9931111
4,1996/97,10070211


### 4.3 Schüler mit sonderpädagogischer Förderung (Inklusion)

In [6]:
inklusion_df = lade_statista_datei(datei_inklusion, "Jahr", "Inklusionsschueler")
print(f"Anzahl Datenzeilen: {len(inklusion_df)}")
inklusion_df.head()

Anzahl Datenzeilen: 15


,Jahr,Inklusionsschueler
0,2009,95475
1,2010,108642
2,2011,121999
3,2012,139605
4,2013,157201


### 4.4 PISA-Ergebnisse

In [7]:
pisa_mathe_df = lade_statista_datei(datei_pisa_mathe, "Jahr", "PISA_Mathematik")
pisa_lesen_df = lade_statista_datei(datei_pisa_lesen, "Jahr", "PISA_Lesekompetenz")
pisa_nawi_df  = lade_statista_datei(datei_pisa_nawi,  "Jahr", "PISA_Naturwissenschaften")

print("PISA Mathematik:")
print(pisa_mathe_df)

PISA Mathematik:
   Jahr  PISA_Mathematik
0  2000              490
1  2003              503
2  2006              504
3  2009              513
4  2012              514
5  2015              506
6  2018              500
7  2022              475


## 5. Datenbereinigung & Transformation

### 5.1. Problem: Schuljahre vs. Kalenderjahre

Die Lehrer- und Schülerdaten sind als **Schuljahre** angegeben (`2010/11`), 
die PISA- und Inklusionsdaten als **Kalenderjahre** (`2010`).

Damit wir die Daten später zusammenführen können, normalisieren wir die Schuljahre 
auf das Anfangsjahr:

   5.1.1. `"2010/11"` → `2010`
   5.1.2. `"2024/25"` → `2024`

In [16]:
# Schuljahr in Kalenderjahr umwandeln (nur die ersten 4 Zeichen behalten)
lehrer_df['Jahr'] = lehrer_df['Schuljahr'].str[:4].astype(int)
schueler_df['Jahr'] = schueler_df['Schuljahr'].str[:4].astype(int)

# Sicherstellen, dass die Anzahlen als ganze Zahlen gespeichert sind
lehrer_df['Anzahl_Lehrer'] = lehrer_df['Anzahl_Lehrer'].astype(int)
schueler_df['Anzahl_Schueler'] = schueler_df['Anzahl_Schueler'].astype(int)
inklusion_df['Inklusionsschueler'] = inklusion_df['Inklusionsschueler'].astype(int)

print("Lehrer-Daten nach Bereinigung:")
print(lehrer_df.head())
print(f"\nDatentypen: {lehrer_df.dtypes.to_dict()}")

Lehrer-Daten nach Bereinigung:
  Schuljahr  Anzahl_Lehrer  Jahr
0   2010/11         763988  2010
1   2011/12         760491  2011
2   2012/13         759136  2012
3   2013/14         752830  2013
4   2014/15         752358  2014

Datentypen: {'Schuljahr': dtype('O'), 'Anzahl_Lehrer': dtype('int64'), 'Jahr': dtype('int64')}


## 6. Neue Kennzahl berechnen: Schüler pro Lehrer

Eine wichtige Kennzahl für unsere Fragestellung: 
**Wie viele Schüler muss ein Lehrer im Durchschnitt betreuen?**

Diese Kennzahl gibt es nicht direkt in den Rohdaten — wir berechnen sie:

$$\text{Schüler pro Lehrer} = \frac{\text{Anzahl Schüler}}{\text{Anzahl Lehrer}}$$

In [9]:
# Lehrer- und Schülerdaten zusammenführen (merge) anhand des Jahres
lehrer_schueler = lehrer_df[['Jahr', 'Anzahl_Lehrer']].merge(
    schueler_df[['Jahr', 'Anzahl_Schueler']], 
    on='Jahr'
)

# Neue Spalte berechnen
lehrer_schueler['Schueler_pro_Lehrer'] = (
    lehrer_schueler['Anzahl_Schueler'] / lehrer_schueler['Anzahl_Lehrer']
).round(2)

print(lehrer_schueler)

    Jahr  Anzahl_Lehrer  Anzahl_Schueler  Schueler_pro_Lehrer
0   2010         763988          8796894                11.51
1   2011         760491          8678196                11.41
2   2012         759136          8556879                11.27
3   2013         752830          8420111                11.18
4   2014         752358          8366666                11.12
5   2015         754726          8335061                11.04
6   2016         759051          8369513                11.03
7   2017         763266          8346707                10.94
8   2018         773280          8330457                10.77
9   2019         782610          8326884                10.64
10  2020         790605          8380767                10.60
11  2021         799314          8436221                10.55
12  2022         820353          8693344                10.60
13  2023         837247          8819489                10.53
14  2024         851193          8914150                10.47


## 7. Alle Daten in einem DataFrame zusammenführen

Jetzt führen wir alle Daten in einer einzigen Tabelle zusammen, mit dem `Jahr` 
als gemeinsamer Schlüssel.

Wir verwenden `merge(..., how='outer')`, damit auch Jahre mit fehlenden Werten 
erhalten bleiben (z.B. hat PISA nur alle 3 Jahre Werte).

In [15]:
# Schrittweise alle DataFrames zusammenführen
gesamt_df = lehrer_schueler.merge(inklusion_df, on='Jahr', how='outer')
gesamt_df = gesamt_df.merge(pisa_mathe_df, on='Jahr', how='outer')
gesamt_df = gesamt_df.merge(pisa_lesen_df, on='Jahr', how='outer')
gesamt_df = gesamt_df.merge(pisa_nawi_df,  on='Jahr', how='outer')

# Nach Jahr sortieren
gesamt_df = gesamt_df.sort_values('Jahr').reset_index(drop=True)

# Auf den relevanten Zeitraum 2009–2024 beschränken
gesamt_df = gesamt_df[gesamt_df['Jahr'] >= 2009].reset_index(drop=True)

gesamt_df

,Jahr,Anzahl_Lehrer,Anzahl_Schueler,Schueler_pro_Lehrer,Inklusionsschueler,PISA_Mathematik,PISA_Lesekompetenz,PISA_Naturwissenschaften
0,2009,NaN,NaN,NaN,95475.0,513.0,497.0,520.0
1,2010,763988.0,8796894.0,11.51,108642.0,NaN,NaN,NaN
2,2011,760491.0,8678196.0,11.41,121999.0,NaN,NaN,NaN
3,2012,759136.0,8556879.0,11.27,139605.0,514.0,508.0,524.0
4,2013,752830.0,8420111.0,11.18,157201.0,NaN,NaN,NaN
5,2014,752358.0,8366666.0,11.12,173392.0,NaN,NaN,NaN
6,2015,754726.0,8335061.0,11.04,194866.0,506.0,509.0,509.0
7,2016,759051.0,8369513.0,11.03,205811.0,NaN,NaN,NaN
8,2017,763266.0,8346707.0,10.94,222485.0,NaN,NaN,NaN
9,2018,773280.0,8330457.0,10.77,235325.0,500.0,498.0,503.0


## 8. Datenqualität prüfen

Bevor wir die Daten an die Visualisierungs-Kollegen weitergeben, prüfen wir:

   8.1. Gibt es fehlende Werte?
   8.2. Sind die Datentypen korrekt?
   8.3. Sind die Werte plausibel?

# Übersicht über die Daten
print("=== ALLGEMEINE INFOS ===")
gesamt_df.info()

print("\n=== FEHLENDE WERTE PRO SPALTE ===")
print(gesamt_df.isna().sum())

print("\n=== STATISTISCHE ZUSAMMENFASSUNG ===")
gesamt_df.describe()

### Interpretation der fehlenden Werte:

   - Bei den **PISA-Spalten** sind viele Werte leer — das ist normal, 
     weil PISA nur alle 3 Jahre durchgeführt wird (2009, 2012, 2015, 2018, 2022)
   
   - Bei **Inklusion**: nur bis 2023 verfügbar
   
   - Bei **Lehrer/Schüler**: vollständig von 2010 bis 2024

Diese Lücken sind **kein Fehler** — die Kollegen können in ihren Visualisierungen 
damit umgehen (z.B. nur die PISA-Jahre für PISA-Grafiken nutzen).

## 9. Bereinigte Daten exportieren

Wir speichern die aufbereiteten Daten in zwei Formaten:

   9.1. **CSV**: einfach lesbar, universell, ideal für Visualisierungen
   9.2. **Excel**: falls jemand die Daten in Excel weiterverarbeiten möchte

In [11]:
# Ordner für aufbereitete Daten erstellen
output_ordner = "Aufbereitete_Daten"
os.makedirs(output_ordner, exist_ok=True)

# Exportieren
csv_pfad   = os.path.join(output_ordner, "gesamtdaten_aufbereitet.csv")
excel_pfad = os.path.join(output_ordner, "gesamtdaten_aufbereitet.xlsx")

gesamt_df.to_csv(csv_pfad, index=False, encoding='utf-8-sig')
gesamt_df.to_excel(excel_pfad, index=False)

print(f"✓ CSV gespeichert:   {csv_pfad}")
print(f"✓ Excel gespeichert: {excel_pfad}")

✓ CSV gespeichert:   Aufbereitete_Daten\gesamtdaten_aufbereitet.csv
✓ Excel gespeichert: Aufbereitete_Daten\gesamtdaten_aufbereitet.xlsx


## 10. Erste Erkenntnisse aus den aufbereiteten Daten

Auch wenn die Visualisierung später durch die Kollegen erfolgt, lassen sich 
schon jetzt **wichtige Trends** erkennen:

In [14]:
# Vergleich Anfang vs. Ende des Zeitraums
print("=== ENTWICKLUNG 2010 → 2024 ===\n")

jahr_start = 2010
jahr_ende  = 2024

zeile_start = gesamt_df[gesamt_df['Jahr'] == jahr_start].iloc[0]
zeile_ende  = gesamt_df[gesamt_df['Jahr'] == jahr_ende].iloc[0]

print(f"Lehrer:               {int(zeile_start['Anzahl_Lehrer']):>10,} → {int(zeile_ende['Anzahl_Lehrer']):>10,}  ({(zeile_ende['Anzahl_Lehrer']/zeile_start['Anzahl_Lehrer']-1)*100:+.1f}%)")
print(f"Schüler:              {int(zeile_start['Anzahl_Schueler']):>10,} → {int(zeile_ende['Anzahl_Schueler']):>10,}  ({(zeile_ende['Anzahl_Schueler']/zeile_start['Anzahl_Schueler']-1)*100:+.1f}%)")
print(f"Schüler pro Lehrer:   {zeile_start['Schueler_pro_Lehrer']:>10.2f} → {zeile_ende['Schueler_pro_Lehrer']:>10.2f}")

# Inklusion separat (geht nur bis 2023)
ink_2010 = gesamt_df[gesamt_df['Jahr']==2010]['Inklusionsschueler'].iloc[0]
ink_2023 = gesamt_df[gesamt_df['Jahr']==2023]['Inklusionsschueler'].iloc[0]
print(f"\nInklusionsschüler 2010 → 2023: {int(ink_2010):,} → {int(ink_2023):,}  ({(ink_2023/ink_2010-1)*100:+.1f}%)")

# PISA Mathematik
pisa_2009 = gesamt_df[gesamt_df['Jahr']==2009]['PISA_Mathematik'].iloc[0]
pisa_2022 = gesamt_df[gesamt_df['Jahr']==2022]['PISA_Mathematik'].iloc[0]
print(f"PISA Mathematik 2009 → 2022:  {pisa_2009:.0f} → {pisa_2022:.0f} Punkte  ({pisa_2022-pisa_2009:+.0f} Punkte)")

=== ENTWICKLUNG 2010 → 2024 ===

Lehrer:                  763,988 →    851,193  (+11.4%)
Schüler:               8,796,894 →  8,914,150  (+1.3%)
Schüler pro Lehrer:        11.51 →      10.47

Inklusionsschüler 2010 → 2023: 108,642 → 263,734  (+142.8%)
PISA Mathematik 2009 → 2022:  513 → 475 Punkte  (-38 Punkte)


### 🎯 Wichtige Befunde

   1. **Lehrer/Schüler-Verhältnis hat sich verbessert** 
      (weniger Schüler pro Lehrer) — auf den ersten Blick gut.

   2. **ABER die Heterogenität ist explodiert**: 
      Inklusionsschüler haben sich seit 2010 mehr als verdoppelt.

   3. **PISA-Ergebnisse sind eingebrochen**: 
      trotz besserem zahlenmäßigen Verhältnis.

### Schlussfolgerung für unsere Fragestellung

Die reine Lehrerzahl reicht nicht aus — die Komplexität der Aufgaben 
(Inklusion, Heterogenität) wächst schneller als die personellen Ressourcen. 

Dies bestätigt die Ausgangsthese: 
**Individuelle Förderung ist unter diesen Bedingungen schwer leistbar.**

Die genaue **visuelle Aufbereitung** dieser Erkenntnisse erfolgt durch 
die Kollegen im nächsten Schritt mit `matplotlib`, `seaborn` oder `plotly`.
